# 스트리밍 알고리즘 2종 구현 및 정확도·메모리 트레이드오프 분석

**Big Data Analytics | Week 07 Assignment**

- 알고리즘: Bloom Filter, Count-Min Sketch
- 데이터셋: MovieLens 1M 형식 합성 데이터 (1,000,209 레코드)
- 구현 방식: Python 표준 라이브러리만 사용, generator 기반 스트림 처리

---
## 실행 순서
1. 환경 준비 & 데이터 생성
2. 알고리즘 구현 (Bloom Filter / Count-Min Sketch)
3. Ground Truth 계산
4. 실험 실행 (파라미터별 비교)
5. 결과 시각화

---
## 1. 환경 준비 및 데이터 생성

In [ ]:
# 필요 라이브러리 임포트 (모두 Python 표준 라이브러리)
import math
import random
import time
import tracemalloc
import hashlib
import struct
import sys
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print(f"Python {sys.version}")
print("모든 라이브러리 로드 완료")

In [ ]:
# ──────────────────────────────────────────────────────────────
# MovieLens 1M 형식 합성 데이터 생성
# 형식: UserID::MovieID::Rating::Timestamp
# 실제 MovieLens 1M과 동일한 레코드 수, 사용자/영화 범위
# ──────────────────────────────────────────────────────────────
DATA_FILE   = 'ratings.dat'
N_RECORDS   = 1_000_209   # MovieLens 1M 원본과 동일
N_USERS     = 6_040
N_MOVIES    = 3_952

random.seed(42)

if not os.path.exists(DATA_FILE):
    print(f"{N_RECORDS:,}개 레코드 생성 중...")
    t0 = time.perf_counter()
    with open(DATA_FILE, 'w') as f:
        for _ in range(N_RECORDS):
            uid    = random.randint(1, N_USERS)
            mid    = random.randint(1, N_MOVIES)
            rating = random.choice([1, 2, 3, 4, 5])
            ts     = random.randint(956_703_932, 1_000_000_000)
            f.write(f"{uid}::{mid}::{rating}::{ts}\n")
    print(f"완료: {time.perf_counter()-t0:.1f}s | "
          f"파일 크기: {os.path.getsize(DATA_FILE)/1024/1024:.1f} MB")
else:
    print(f"기존 파일 사용: {os.path.getsize(DATA_FILE)/1024/1024:.1f} MB")

# 첫 5줄 확인
with open(DATA_FILE) as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 4:
            break

---
## 2. 스트림 생성기 (Generator 기반)

전체 데이터를 메모리에 올리지 않고 **한 줄씩** 처리하는 방식입니다.  
실제 Kafka, Kinesis 등 스트리밍 플랫폼의 처리 패턴과 동일합니다.

In [ ]:
def stream_records(filepath):
    """파일을 한 줄씩 읽어 (user_id, movie_id, rating, ts) 튜플 yield"""
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split('::')
            if len(parts) == 4:
                yield parts[0], parts[1], parts[2], parts[3]

def stream_movie_ids(filepath):
    """movie_id만 스트림으로 추출"""
    for _, mid, _, _ in stream_records(filepath):
        yield mid

def stream_user_ids(filepath):
    """user_id만 스트림으로 추출"""
    for uid, _, _, _ in stream_records(filepath):
        yield uid

# 확인: 첫 3개 레코드
gen = stream_records(DATA_FILE)
for i in range(3):
    print(next(gen))

---
## 3. 알고리즘 구현

### 3-1. Bloom Filter

**핵심 수식**
- 비트 배열 크기: `m = -(n × ln p) / (ln 2)²`
- 해시 함수 수: `k = (m/n) × ln 2`
- 이중 해싱: `hash_i(x) = (h1(x) + i × h2(x)) mod m`

In [ ]:
class BloomFilter:
    """
    Bloom Filter 직접 구현

    Parameters
    ----------
    capacity   : 예상 삽입 원소 수 (n)
    error_rate : 목표 False Positive Rate (p)

    특성
    ----
    - False Negative 절대 없음 (확실한 보장)
    - False Positive 발생 가능 (error_rate 이하로 제어)
    - 삭제 불가 (표준 구현)
    """

    def __init__(self, capacity, error_rate=0.01):
        self.capacity   = capacity
        self.error_rate = error_rate

        # 최적 비트 배열 크기: m = -(n * ln(p)) / (ln(2))^2
        self.bit_size = int(
            -capacity * math.log(error_rate) / (math.log(2) ** 2)
        )
        # 최적 해시 함수 수: k = (m/n) * ln(2)
        self.hash_count = max(
            1, int((self.bit_size / capacity) * math.log(2))
        )
        # 비트 배열 (bytearray: 1바이트 = 8비트)
        self.bit_array = bytearray(math.ceil(self.bit_size / 8))
        self.insert_count = 0

    # ── 이중 해싱으로 k개 비트 위치 생성 ────────────────────────
    def _hashes(self, item):
        b  = item.encode('utf-8') if isinstance(item, str) else str(item).encode()
        h1 = int(hashlib.md5(b).hexdigest(), 16)
        h2 = int(hashlib.sha256(b).hexdigest(), 16)
        for i in range(self.hash_count):
            yield (h1 + i * h2) % self.bit_size

    # ── 삽입: k개 비트 위치를 1로 설정 ──────────────────────────
    def add(self, item):
        for bit_pos in self._hashes(item):
            byte_idx, bit_idx = divmod(bit_pos, 8)
            self.bit_array[byte_idx] |= (1 << bit_idx)
        self.insert_count += 1

    # ── 조회: k개 비트 위치가 모두 1인지 확인 ───────────────────
    def __contains__(self, item):
        return all(
            self.bit_array[byte_idx] & (1 << bit_idx)
            for bit_pos in self._hashes(item)
            for byte_idx, bit_idx in [divmod(bit_pos, 8)]
        )

    def memory_bytes(self):
        return len(self.bit_array)

    def __repr__(self):
        return (f"BloomFilter(capacity={self.capacity:,}, "
                f"error_rate={self.error_rate}, "
                f"bit_size={self.bit_size:,}, "
                f"hash_count={self.hash_count}, "
                f"memory={self.memory_bytes()/1024:.1f}KB)")


# 동작 확인
bf_test = BloomFilter(capacity=1000, error_rate=0.01)
for item in ['apple', 'banana', 'cherry']:
    bf_test.add(item)

print("[Bloom Filter 동작 확인]")
print(f"  'apple'   in BF: {'apple'   in bf_test}  (기대: True)")
print(f"  'banana'  in BF: {'banana'  in bf_test}  (기대: True)")
print(f"  'durian'  in BF: {'durian'  in bf_test}  (기대: False)")
print(f"\n{bf_test}")

### 3-2. Count-Min Sketch

**핵심 수식**
- 오차 보장: `추정치 ≤ 실제값 + ε×N` (확률 `1-δ`)
- 이론 오차: `ε = e/w`, 실패 확률: `δ = e^(-d)`
- 쿼리: `min(table[i][hash_i(x)])` for i in 0..d-1

In [ ]:
class CountMinSketch:
    """
    Count-Min Sketch 직접 구현

    Parameters
    ----------
    width : 행렬 열 수 (w) — 클수록 오차 감소
    depth : 행렬 행 수 (d) — 클수록 실패 확률 감소

    특성
    ----
    - 과추정(over-count)만 발생: 실제값 ≤ 추정값
    - 이론 오차 ε = e/w, 실패 확률 δ = e^(-d)
    - 병합 가능 → 분산 환경에 적합
    """

    def __init__(self, width, depth):
        self.width = width
        self.depth = depth
        # d × w 정수 행렬 초기화
        self.table = [[0] * width for _ in range(depth)]
        # 각 행마다 독립적인 랜덤 seed
        self.seeds = [random.randint(0, 2**31 - 1) for _ in range(depth)]

    # ── seed 기반 독립 해시 ───────────────────────────────────────
    def _hash(self, item, seed):
        b = item.encode('utf-8') if isinstance(item, str) else str(item).encode()
        h = hashlib.md5(b + struct.pack('I', seed)).hexdigest()
        return int(h, 16) % self.width

    # ── 삽입: d개 행의 해당 열 카운터 +1 ────────────────────────
    def add(self, item, count=1):
        for i, seed in enumerate(self.seeds):
            j = self._hash(item, seed)
            self.table[i][j] += count

    # ── 쿼리: d개 행의 값 중 최솟값 반환 ────────────────────────
    def query(self, item):
        return min(
            self.table[i][self._hash(item, self.seeds[i])]
            for i in range(self.depth)
        )

    def memory_bytes(self):
        return self.width * self.depth * 8   # int64 기준

    def epsilon(self):
        """이론적 오차 상한 ε = e / w"""
        return math.e / self.width

    def delta(self):
        """이론적 실패 확률 δ = e^(-d)"""
        return math.exp(-self.depth)

    def __repr__(self):
        return (f"CountMinSketch(width={self.width}, depth={self.depth}, "
                f"ε={self.epsilon():.4f}, δ={self.delta():.4f}, "
                f"memory={self.memory_bytes()/1024:.1f}KB)")


# 동작 확인
cms_test = CountMinSketch(width=1000, depth=5)
items = ['apple'] * 10 + ['banana'] * 5 + ['cherry'] * 3
random.shuffle(items)
for item in items:
    cms_test.add(item)

print("[Count-Min Sketch 동작 확인]")
print(f"  'apple'  추정: {cms_test.query('apple')}  (실제: 10)")
print(f"  'banana' 추정: {cms_test.query('banana')}  (실제: 5)")
print(f"  'cherry' 추정: {cms_test.query('cherry')}  (실제: 3)")
print(f"\n{cms_test}")

---
## 4. Ground Truth 계산

스트리밍 알고리즘 결과와 비교하기 위한 정확한 기준값을 계산합니다.

| 알고리즘 | Ground Truth 방법 |
|---|---|
| Bloom Filter | Python `set()` 기반 실제 멤버십 |
| Count-Min Sketch | Python `defaultdict(int)` 기반 정확한 빈도 |

In [ ]:
print("Ground Truth 계산 중 (단일 패스)...")

gt_movie_set   = set()
gt_movie_count = defaultdict(int)
gt_user_set    = set()
total_records  = 0

t_gt0 = time.perf_counter()
for uid, mid, rating, ts in stream_records(DATA_FILE):
    gt_movie_set.add(mid)
    gt_movie_count[mid] += 1
    gt_user_set.add(uid)
    total_records += 1
t_gt1 = time.perf_counter()

GT_UNIQUE_MOVIES = len(gt_movie_set)
GT_UNIQUE_USERS  = len(gt_user_set)
GT_TOTAL         = total_records
GT_TIME          = t_gt1 - t_gt0

# Ground Truth 메모리 측정
GT_SET_MEM_KB  = sys.getsizeof(gt_movie_set) / 1024
GT_DICT_MEM_KB = (sys.getsizeof(gt_movie_count)
                  + sum(sys.getsizeof(k) + sys.getsizeof(v)
                        for k, v in gt_movie_count.items())) / 1024

top20_true = sorted(gt_movie_count.items(), key=lambda x: -x[1])[:20]

# 테스트 셋 구성
TEST_IN  = list(gt_movie_set)[:200]            # 실제 존재 영화 200개
TEST_OUT = [str(i) for i in range(4000, 4200)] # 존재하지 않는 영화 200개

print(f"\n{'='*45}")
print(f"  총 레코드 수   : {GT_TOTAL:>12,}")
print(f"  고유 영화 수   : {GT_UNIQUE_MOVIES:>12,}")
print(f"  고유 사용자 수 : {GT_UNIQUE_USERS:>12,}")
print(f"  처리 시간      : {GT_TIME:>11.2f}s")
print(f"  set 메모리     : {GT_SET_MEM_KB:>11.1f} KB")
print(f"  dict 메모리    : {GT_DICT_MEM_KB:>11.1f} KB")
print(f"{'='*45}")
print(f"\n상위 5개 영화 (Ground Truth):")
for mid, cnt in top20_true[:5]:
    print(f"  movie_id={mid:>6}  count={cnt:,}")

---
## 5. Bloom Filter 실험 (파라미터별 비교)

비교 파라미터: `capacity` (예상 원소 수), `error_rate` (목표 FPR)

In [ ]:
BF_CONFIGS = [
    (100_000, 0.10, "cap=100K, ε=10%"),
    (100_000, 0.01, "cap=100K, ε=1%"),
    (100_000, 0.001,"cap=100K, ε=0.1%"),
    (500_000, 0.01, "cap=500K, ε=1%"),
]

BF_RESULTS = []

for cap, err, label in BF_CONFIGS:
    bf = BloomFilter(cap, err)

    # ── 스트림 처리 + 메모리/시간 측정 ──
    tracemalloc.start()
    t0 = time.perf_counter()
    cnt = 0
    for mid in stream_movie_ids(DATA_FILE):
        bf.add(mid)
        cnt += 1
    t1 = time.perf_counter()
    _, peak_kb = [x/1024 for x in tracemalloc.get_traced_memory()]
    tracemalloc.stop()

    # ── 정확도 측정 (vs Ground Truth) ──
    fp  = sum(1 for x in TEST_OUT if x in bf)   # False Positive
    fn  = sum(1 for x in TEST_IN  if x not in bf)  # False Negative
    fpr = fp / len(TEST_OUT)
    fnr = fn / len(TEST_IN)

    r = dict(
        label=label, cap=cap, err=err,
        bit_size=bf.bit_size, hash_count=bf.hash_count,
        mem_kb=bf.memory_bytes()/1024,
        time_s=t1-t0, tp=cnt/(t1-t0),
        fpr=fpr, fnr=fnr, fp=fp, fn=fn
    )
    BF_RESULTS.append(r)
    print(f"[{label}]")
    print(f"  비트 배열: {bf.bit_size:,} bits | 해시 함수: {bf.hash_count}개")
    print(f"  메모리: {r['mem_kb']:.1f} KB | 처리 시간: {r['time_s']:.2f}s | 처리량: {r['tp']:,.0f} rec/s")
    print(f"  FPR: {fpr:.4f} | FP: {fp} | FN: {fn}")
    print()

---
## 6. Count-Min Sketch 실험 (파라미터별 비교)

비교 파라미터: `width` (w, 열 수), `depth` (d, 행 수)

In [ ]:
CMS_CONFIGS = [
    (100,  3, "w=100,  d=3"),
    (500,  3, "w=500,  d=3"),
    (1000, 3, "w=1000, d=3"),
    (1000, 5, "w=1000, d=5"),
    (5000, 5, "w=5000, d=5"),
]

CMS_RESULTS = []

for w, d, label in CMS_CONFIGS:
    cms = CountMinSketch(w, d)

    # ── 스트림 처리 + 시간 측정 ──
    t0 = time.perf_counter()
    cnt = 0
    for mid in stream_movie_ids(DATA_FILE):
        cms.add(mid)
        cnt += 1
    t1 = time.perf_counter()

    # ── 정확도: 상위 20개 영화에 대한 오차 ──
    errs_top20 = []
    for mid, true_cnt in top20_true:
        est = cms.query(mid)
        errs_top20.append(abs(est - true_cnt) / true_cnt * 100)

    # ── 정확도: 전체 샘플 100개 ──
    sample_keys = list(gt_movie_count.keys())[:100]
    errs_all = [
        abs(cms.query(k) - gt_movie_count[k]) / gt_movie_count[k] * 100
        for k in sample_keys
    ]

    r = dict(
        label=label, w=w, d=d,
        mem_kb=cms.memory_bytes()/1024,
        time_s=t1-t0, tp=cnt/(t1-t0),
        avg_err_top20=sum(errs_top20)/len(errs_top20),
        max_err_top20=max(errs_top20),
        overall_err=sum(errs_all)/len(errs_all),
        epsilon=cms.epsilon(), delta=cms.delta()
    )
    CMS_RESULTS.append(r)
    print(f"[{label}]")
    print(f"  메모리: {r['mem_kb']:.1f} KB | 처리 시간: {r['time_s']:.2f}s | 처리량: {r['tp']:,.0f} rec/s")
    print(f"  이론 ε={cms.epsilon():.4f}, δ={cms.delta():.4f}")
    print(f"  Top-20 평균 오차: {r['avg_err_top20']:.2f}% | 최대: {r['max_err_top20']:.2f}%")
    print(f"  전체 평균 오차:   {r['overall_err']:.2f}%")
    print()

---
## 7. Ground Truth vs 스트리밍 알고리즘 비교 요약

In [ ]:
print("="*60)
print("Ground Truth vs Bloom Filter 비교")
print("="*60)
print(f"  GT (set 기반) 메모리  : {GT_SET_MEM_KB:.1f} KB")
print(f"  GT 처리 시간          : {GT_TIME:.2f}s")
print()
for r in BF_RESULTS:
    saving = (1 - r['mem_kb']/GT_SET_MEM_KB) * 100
    print(f"  [{r['label']}]")
    print(f"    메모리: {r['mem_kb']:.1f} KB  (GT 대비 {saving:.1f}% 절감)")
    print(f"    FPR: {r['fpr']:.4f}  FN: {r['fn']}")

print()
print("="*60)
print("Ground Truth vs Count-Min Sketch 비교")
print("="*60)
print(f"  GT (dict 기반) 메모리 : {GT_DICT_MEM_KB:.1f} KB")
print()
for r in CMS_RESULTS:
    saving = (1 - r['mem_kb']/GT_DICT_MEM_KB) * 100
    print(f"  [{r['label']}]")
    print(f"    메모리: {r['mem_kb']:.1f} KB  (GT 대비 {saving:.1f}% 절감)")
    print(f"    Top-20 평균 오차: {r['avg_err_top20']:.2f}%  전체 평균: {r['overall_err']:.2f}%")

---
## 8. 결과 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('스트리밍 알고리즘 실험 결과 요약', fontsize=15, fontweight='bold', y=1.01)

# ── ① Bloom Filter: 메모리 vs 처리량 ──────────────────────────
ax = axes[0, 0]
labels_bf = [r['label'].replace(', ', '\n') for r in BF_RESULTS]
mem_bf    = [r['mem_kb'] for r in BF_RESULTS]
tp_bf     = [r['tp']/1000 for r in BF_RESULTS]
x = np.arange(len(labels_bf))
b1 = ax.bar(x - 0.2, mem_bf, 0.4, label='Memory (KB)', color='#2E86AB', alpha=0.85)
ax2 = ax.twinx()
b2 = ax2.bar(x + 0.2, tp_bf, 0.4, label='Throughput (K/s)', color='#F4A261', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels_bf, fontsize=8)
ax.set_ylabel('Memory (KB)', color='#2E86AB')
ax2.set_ylabel('Throughput (K rec/s)', color='#F4A261')
ax.set_title('Bloom Filter: Memory vs Throughput', fontweight='bold')
lines = [mpatches.Patch(color='#2E86AB', label='Memory (KB)'),
         mpatches.Patch(color='#F4A261', label='Throughput (K/s)')]
ax.legend(handles=lines, loc='upper left', fontsize=8)

# ── ② CMS: 설정별 오차율 ──────────────────────────────────────
ax = axes[0, 1]
labels_cms = [r['label'] for r in CMS_RESULTS]
errs20     = [r['avg_err_top20'] for r in CMS_RESULTS]
errs_all   = [r['overall_err'] for r in CMS_RESULTS]
x = np.arange(len(labels_cms))
ax.bar(x - 0.2, errs20,   0.4, label='Top-20 Avg Error (%)', color='#E84855', alpha=0.85)
ax.bar(x + 0.2, errs_all, 0.4, label='Overall Avg Error (%)', color='#2A9D8F', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels_cms, fontsize=8, rotation=15)
ax.set_ylabel('Relative Error (%)')
ax.set_title('Count-Min Sketch: Error Rate by Config', fontweight='bold')
ax.legend(fontsize=8)

# ── ③ CMS: Memory–Accuracy Trade-off (scatter) ────────────────
ax = axes[1, 0]
for r in CMS_RESULTS:
    ax.scatter(r['mem_kb'], r['overall_err'], s=100, color='#1E3A5F', zorder=5)
    ax.annotate(r['label'], (r['mem_kb'], r['overall_err']),
                textcoords='offset points', xytext=(6, 3), fontsize=8)
ax.set_xlabel('Memory (KB)')
ax.set_ylabel('Overall Avg Error (%)')
ax.set_title('CMS: Memory–Accuracy Trade-off', fontweight='bold')
ax.axhline(y=10, color='#E84855', linestyle='--', alpha=0.5, label='10% threshold')
ax.legend(fontsize=8)

# ── ④ 전체 처리량 비교 ────────────────────────────────────────
ax = axes[1, 1]
all_labels = [f'BF\n{r["label"].split(",")[1].strip()}' for r in BF_RESULTS] + \
             [f'CMS\n{r["label"]}' for r in CMS_RESULTS]
all_tps    = [r['tp']/1000 for r in BF_RESULTS] + [r['tp']/1000 for r in CMS_RESULTS]
all_colors = ['#2E86AB']*len(BF_RESULTS) + ['#F4A261']*len(CMS_RESULTS)
ax.bar(range(len(all_labels)), all_tps, color=all_colors, alpha=0.85)
ax.set_xticks(range(len(all_labels)))
ax.set_xticklabels(all_labels, fontsize=7.5)
ax.set_ylabel('Throughput (K rec/s)')
ax.set_title('Throughput Comparison (All Configs)', fontweight='bold')
bf_p  = mpatches.Patch(color='#2E86AB', label='Bloom Filter')
cms_p = mpatches.Patch(color='#F4A261', label='Count-Min Sketch')
ax.legend(handles=[bf_p, cms_p], fontsize=8)

plt.tight_layout()
plt.savefig('experiment_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("차트 저장: experiment_results.png")

---
## 9. 최종 분석 질문 답변

In [ ]:
print("Q1. 정확도와 메모리 사이에는 어떤 trade-off가 있었는가?")
print("-"*60)
print("[Bloom Filter]")
for r in BF_RESULTS:
    print(f"  {r['label']:20s} → 메모리: {r['mem_kb']:6.1f} KB | FPR: {r['fpr']:.4f}")
print()
print("[Count-Min Sketch]")
for r in CMS_RESULTS:
    print(f"  {r['label']:16s} → 메모리: {r['mem_kb']:6.1f} KB | 전체 평균 오차: {r['overall_err']:7.2f}%")

print()
print("결론: BF는 메모리 증가 시 오율 감소 (로그 선형 관계)")
print("      CMS는 임계치 기반 비선형 개선 (w=5000 이상에서 급격히 개선)")

print()
print("Q2. 파라미터 증가가 항상 성능 향상으로 이어졌는가?")
print("-"*60)
best_tp_bf  = max(BF_RESULTS,  key=lambda r: r['tp'])
worst_tp_bf = min(BF_RESULTS,  key=lambda r: r['tp'])
print(f"  BF 최고 처리량: {best_tp_bf['label']} → {best_tp_bf['tp']:,.0f} rec/s")
print(f"  BF 최저 처리량: {worst_tp_bf['label']} → {worst_tp_bf['tp']:,.0f} rec/s")
print(f"  → 해시 함수 수 증가로 처리량 {(1 - worst_tp_bf['tp']/best_tp_bf['tp'])*100:.0f}% 감소")

print()
print("Q3. 어떤 알고리즘이 가장 실용적인가?")
print("-"*60)
print("  멤버십 판정 목적 → Bloom Filter (FN=0 보장, 구현 단순, 빠른 처리)")
print("  빈도 추정 목적   → Count-Min Sketch (w=5000,d=5 이상 권장)")

print()
print("Q4. 실제 서비스 로그 분석에 적용한다면?")
print("-"*60)
print("  BF  → 중복 이벤트 필터링, 악성 IP 실시간 탐지")
print("  CMS → Top-K 인기 콘텐츠 집계, Rate Limiting, 이상 트래픽 탐지")
print("  → Spark Structured Streaming의 foreachBatch에 두 알고리즘 조합 적용 권장")